# Load Model

In [1]:
import re
import os
import random
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
# from google.colab import userdata
from peft import PeftModel
import time
import json
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
token = user_secrets.get_secret("HF_TOKEN")

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Colab only
# token = userdata.get('HF_TOKEN')
# if not token:
#     raise EnvironmentError("Set HF_TOKEN environment variable")

login(token=token)

MODEL_ID = "CohereLabs/tiny-aya-base"
ADAPTER_REPO = "legesher/language-decoded-lora"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def load_model(merge_adapter=True):
    # Step 1: load base model
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )

    # # Step 2: if no adapter requested, just return base model
    # if adapter_subfolder is None:
    #     model = base_model
    #     print("Loaded base model only.")

    # Step 3: load LoRA adapter from the repo + chosen subfolder

    print(f"Loading adapter")
    model = PeftModel.from_pretrained(
        base_model,
        ADAPTER_REPO
    )

    # Step 4: optionally merge adapter weights into base model
    if merge_adapter:
        print("Merging adapter into base model for inference...")
        model = model.merge_and_unload()
    else:
        print("Keeping model as PEFT model (base + adapter not merged).")

    model.eval()
    return model

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

# Load adapters

In [2]:
model = load_model(merge_adapter = True)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Loading adapter


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/121M [00:00<?, ?B/s]

Merging adapter into base model for inference...


Shared Helpers

In [3]:
def generate_text(prompt: str, max_new_tokens: int = 80) -> str:
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(next(model.parameters()).device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_text[len(prompt):].strip()

# Load Datasets

MGSM

In [4]:
mgsm_zh = pd.read_csv("https://raw.githubusercontent.com/google-research-datasets/MGSM-Rev2/main/MGSM-Rev2/mgsm_zh.tsv", sep="\t", header=None)
mgsm_zh = mgsm_zh.rename(columns={0: "Question", 1: "Answer"})

mgsm_es = pd.read_csv("https://raw.githubusercontent.com/google-research-datasets/MGSM-Rev2/main/MGSM-Rev2/mgsm_es.tsv", sep="\t", header=None)
mgsm_es = mgsm_es.rename(columns={0: "Question", 1: "Answer"})

mgsm_ur_ds = load_dataset("large-traversaal/mgsm_urdu_final")
mgsm_ur = mgsm_ur_ds["test"].to_pandas()
mgsm_ur = mgsm_ur.drop(columns=["question", "answer", "urdu_answer", "equation_solution"])
mgsm_ur = mgsm_ur.rename(columns={"urdu_question": "Question", "answer_number": "Answer"})

README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


data/train-00000-of-00001.parquet:   0%|          | 0.00/9.10k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/87.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/250 [00:00<?, ? examples/s]

XNLI

In [5]:
xnli_zh = load_dataset("facebook/xnli", "zh")["test"].to_pandas()
xnli_es = load_dataset("facebook/xnli", "es")["test"].to_pandas()
xnli_ur = load_dataset("facebook/xnli", "ur")["test"].to_pandas()

README.md: 0.00B [00:00, ?B/s]

zh/train-00000-of-00001.parquet:   0%|          | 0.00/47.8M [00:00<?, ?B/s]

zh/test-00000-of-00001.parquet:   0%|          | 0.00/310k [00:00<?, ?B/s]

zh/validation-00000-of-00001.parquet:   0%|          | 0.00/157k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

...

ur/test-00000-of-00001.parquet:   0%|          | 0.00/428k [00:00<?, ?B/s]

ur/validation-00000-of-00001.parquet:   0%|          | 0.00/216k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5010 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2490 [00:00<?, ? examples/s]

CSQA

In [6]:
csqa_es = load_dataset("INK-USC/xcsr", "X-CSQA-es")["validation"].to_pandas()
csqa_zh = load_dataset("INK-USC/xcsr", "X-CSQA-zh")["validation"].to_pandas()
csqa_ur = load_dataset("INK-USC/xcsr", "X-CSQA-ur")["validation"].to_pandas()

def prepare_csqa(df):
    df = df[["question", "answerKey"]].copy()
    df["stem"] = df["question"].apply(lambda x: x["stem"])
    df["choice_texts"] = df["question"].apply(lambda x: list(x["choices"]["text"]))

    df["A"] = df["choice_texts"].apply(lambda x: x[0])
    df["B"] = df["choice_texts"].apply(lambda x: x[1])
    df["C"] = df["choice_texts"].apply(lambda x: x[2])
    df["D"] = df["choice_texts"].apply(lambda x: x[3])
    df["E"] = df["choice_texts"].apply(lambda x: x[4])

    return df[["stem", "A", "B", "C", "D", "E", "answerKey"]]

csqa_es = prepare_csqa(csqa_es)
csqa_zh = prepare_csqa(csqa_zh)
csqa_ur = prepare_csqa(csqa_ur)

README.md: 0.00B [00:00, ?B/s]

X-CSQA-es/test-00000-of-00001.parquet:   0%|          | 0.00/124k [00:00<?, ?B/s]

X-CSQA-es/validation-00000-of-00001.parq(…):   0%|          | 0.00/115k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1074 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

X-CSQA-ur/test-00000-of-00001.parquet:   0%|          | 0.00/139k [00:00<?, ?B/s]

X-CSQA-ur/validation-00000-of-00001.parq(…):   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1074 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

## English Benchmark Data (for catastrophic forgetting check)

In [ ]:
# English benchmark data — evaluate to detect catastrophic forgetting
# (Did fine-tuning on multilingual code hurt English performance?)

# MGSM English
mgsm_en = pd.read_csv("https://raw.githubusercontent.com/google-research-datasets/MGSM-Rev2/main/MGSM-Rev2/mgsm_en.tsv", sep="\t", header=None)
mgsm_en = mgsm_en.rename(columns={0: "Question", 1: "Answer"})

# XNLI English
xnli_en = load_dataset("facebook/xnli", "en")["test"].to_pandas()

# CSQA English
csqa_en = load_dataset("INK-USC/xcsr", "X-CSQA-en")["validation"].to_pandas()
csqa_en = prepare_csqa(csqa_en)

MGSM Functions

In [7]:
def build_mgsm_prompt(question: str, lang) -> str:
    templates = {
        "en": f"""Solve the following math word problem.
Give the final answer as a number only.

Question: {question}
Answer:""",
        "es": f"""Resuelve el siguiente problema matemático.
Da la respuesta final solo como un número.

Pregunta: {question}
Respuesta:""",
        "zh": f"""请解答下面的数学应用题。
最终答案只输出数字。

题目: {question}
答案:""",
        "ur": f"""مندرجہ ذیل ریاضی کے سوال کو حل کریں۔
صرف عددی شکل میں آخری جواب دیں۔

سوال: {question}
جواب:"""
    }
    return templates[lang]

def extract_number(text: str):
    matches = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return matches[-1] if matches else None

def normalize_number_string(x):
    x = str(x).strip().replace(",", "")
    matches = re.findall(r"-?\d+(?:\.\d+)?", x)
    return matches[-1] if matches else None

def evaluate_mgsm(df: pd.DataFrame, lang, n_samples=None):
    total, correct = 0, 0
    rows = []

    eval_df = df if n_samples is None else df.head(n_samples)

    for _, row in eval_df.iterrows():
        prompt = build_mgsm_prompt(row["Question"], lang)
        output = generate_text(prompt, max_new_tokens=80)
        pred = extract_number(output)
        gold = normalize_number_string(row["Answer"])

        is_correct = pred == gold
        correct += int(is_correct)
        total += 1

        rows.append({
            "question": row["Question"],
            "raw_output": output,
            "pred": pred,
            "gold": gold,
            "correct": is_correct
        })

    return correct / total if total else 0.0, pd.DataFrame(rows)

XNLI Functions

In [8]:
LABEL_MAP = {
    0: "entailment",
    1: "neutral",
    2: "contradiction"
}

def build_xnli_prompt(row, lang):
    templates = {
        "en": f"""Read the premise and hypothesis below.
Decide whether the hypothesis is entailed by the premise, contradicted by the premise, or neither.
Reply with only one word: entailment, contradiction, or neutral.

Premise: {row['premise']}
Hypothesis: {row['hypothesis']}

Answer:""",

        "es": f"""Lee la premisa y la hipótesis a continuación.
Decide si la hipótesis se sigue de la premisa, la contradice, o no es ninguna de las dos.
Responde con solo una palabra: entailment, contradiction, o neutral.

Premisa: {row['premise']}
Hipótesis: {row['hypothesis']}

Respuesta:""",

        "zh": f"""请阅读下面的前提和假设。
判断假设是否被前提蕴含、与前提矛盾，或两者都不是。
只回答一个词：entailment、contradiction 或 neutral。

前提: {row['premise']}
假设: {row['hypothesis']}

答案:""",

        "ur": f"""نیچے دی گئی premise اور hypothesis کو پڑھیں۔
فیصلہ کریں کہ hypothesis، premise سے لازم آتی ہے، اس کی تردید کرتی ہے، یا دونوں میں سے کوئی نہیں۔
صرف ایک لفظ میں جواب دیں: entailment، contradiction، یا neutral۔

Premise: {row['premise']}
Hypothesis: {row['hypothesis']}

جواب:"""
    }
    return templates[lang]


NATIVE_LABEL_MAP = {
    # Chinese
    "蕴含": "entailment", "蕴涵": "entailment",
    "矛盾": "contradiction",
    "中立": "neutral",
    # Spanish
    "implicación": "entailment", "implicacion": "entailment",
    "contradicción": "contradiction", "contradiccion": "contradiction",
    # Urdu
    "لازمی": "entailment",
    "تردید": "contradiction",
    "غیرجانبدار": "neutral",
}

def extract_xnli_label(text: str):
    text_lower = text.strip().lower()
    # Try English labels first
    for label in ["entailment", "contradiction", "neutral"]:
        if re.search(rf"\b{label}\b", text_lower):
            return label
    # Try native language labels
    for native, english in NATIVE_LABEL_MAP.items():
        if native in text.strip():
            return english
    return None


def normalize_xnli_gold(label):
    if isinstance(label, str):
        return label.strip().lower()
    return LABEL_MAP.get(int(label), None)

def evaluate_xnli(df: pd.DataFrame, lang, n_samples=None):
    total, correct = 0, 0
    rows = []

    eval_df = df if n_samples is None else df.head(n_samples)

    for _, row in eval_df.iterrows():
        prompt = build_xnli_prompt(row, lang)
        output = generate_text(prompt, max_new_tokens=10)
        pred = extract_xnli_label(output)
        gold = normalize_xnli_gold(row["label"])

        is_correct = pred == gold
        correct += int(is_correct)
        total += 1

        rows.append({
            "premise": row["premise"],
            "hypothesis": row["hypothesis"],
            "raw_output": output,
            "pred": pred,
            "gold": gold,
            "correct": is_correct
        })

    return correct / total if total else 0.0, pd.DataFrame(rows)

CSQA Functions

In [9]:
def build_csqa_prompt(row, lang):
    templates = {
        "en": f"""Choose the best answer to the following commonsense question.
Reply with only one letter: A, B, C, D, or E.

Question: {row['stem']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}

Answer:""",

        "es": f"""Elige la mejor respuesta para la siguiente pregunta de sentido común.
Responde solo con una letra: A, B, C, D o E.

Pregunta: {row['stem']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}

Respuesta:""",

        "zh": f"""请选择下面这个常识问题的最佳答案。
只回答一个字母：A、B、C、D 或 E。

问题: {row['stem']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}

答案:""",

        "ur": f"""نیچے دیے گئے عمومی فہم کے سوال کا بہترین جواب منتخب کریں۔
صرف ایک حرف میں جواب دیں: A، B، C، D، یا E۔

سوال: {row['stem']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}

جواب:"""
    }
    return templates[lang]

def extract_choice(text: str):
    text = text.strip().upper()

    match = re.search(r"\b([ABCDE])\b", text)
    if match:
        return match.group(1)

    match = re.search(r"ANSWER\s*[:\-]?\s*([ABCDE])", text)
    if match:
        return match.group(1)

    return None

def evaluate_csqa(df: pd.DataFrame, lang, n_samples=None):
    total, correct = 0, 0
    rows = []

    eval_df = df if n_samples is None else df.head(n_samples)

    for _, row in eval_df.iterrows():
        prompt = build_csqa_prompt(row, lang)
        output = generate_text(prompt, max_new_tokens=10)
        pred = extract_choice(output)
        gold = str(row["answerKey"]).strip().upper()

        is_correct = pred == gold
        correct += int(is_correct)
        total += 1

        rows.append({
            "stem": row["stem"],
            "raw_output": output,
            "pred": pred,
            "gold": gold,
            "correct": is_correct
        })

    return correct / total if total else 0.0, pd.DataFrame(rows)

# Evals

In [10]:
def results_to_jsonable(results):
    jsonable = {
        "summary": results["summary"],
        "timing": results["timing"],
    }

    for key, value in results.items():
        if key in ["summary", "timing"]:
            continue
        if value is None:
            jsonable[key] = None
        else:
            jsonable[key] = value.to_dict(orient="records")

    return jsonable

All Evals

In [11]:
from huggingface_hub import HfApi
import json

cond = "condition-1-en"


def eval_english_prompts():
    summary = {}

    summary["mgsm_zh_acc"], res_mgsm_zh = evaluate_mgsm(mgsm_zh, lang='en', n_samples=None)
    summary["mgsm_es_acc"], res_mgsm_es = evaluate_mgsm(mgsm_es, lang='en', n_samples=None)
    summary["mgsm_ur_acc"], res_mgsm_ur = evaluate_mgsm(mgsm_ur, lang='en', n_samples=None)
    print('MGSM Done - EN')

    summary["xnli_zh_acc"], res_xnli_zh = evaluate_xnli(xnli_zh, lang='en', n_samples=None)
    summary["xnli_es_acc"], res_xnli_es = evaluate_xnli(xnli_es, lang='en', n_samples=None)
    summary["xnli_ur_acc"], res_xnli_ur = evaluate_xnli(xnli_ur, lang='en', n_samples=None)
    print('XNLI Done - EN')

    summary["csqa_zh_acc"], res_csqa_zh = evaluate_csqa(csqa_zh, lang='en', n_samples=None)
    summary["csqa_es_acc"], res_csqa_es = evaluate_csqa(csqa_es, lang='en', n_samples=None)
    summary["csqa_ur_acc"], res_csqa_ur = evaluate_csqa(csqa_ur, lang='en', n_samples=None)
    print('CSQA Done - EN')

    # English data — catastrophic forgetting check
    summary["mgsm_en_acc"], res_mgsm_en = evaluate_mgsm(mgsm_en, lang='en', n_samples=None)
    summary["xnli_en_acc"], res_xnli_en = evaluate_xnli(xnli_en, lang='en', n_samples=None)
    summary["csqa_en_acc"], res_csqa_en = evaluate_csqa(csqa_en, lang='en', n_samples=None)
    print('English data Done - forgetting check')

    return {
        "summary": summary,
        "mgsm_zh": res_mgsm_zh,
        "mgsm_es": res_mgsm_es,
        "mgsm_ur": res_mgsm_ur,
        "mgsm_en": res_mgsm_en,
        "xnli_zh": res_xnli_zh,
        "xnli_es": res_xnli_es,
        "xnli_ur": res_xnli_ur,
        "xnli_en": res_xnli_en,
        "csqa_zh": res_csqa_zh,
        "csqa_es": res_csqa_es,
        "csqa_ur": res_csqa_ur,
        "csqa_en": res_csqa_en,
    }


def eval_native_prompts():
    summary = {}

    summary["mgsm_zh_acc"], res_mgsm_zh = evaluate_mgsm(mgsm_zh, lang='zh', n_samples=None)
    summary["mgsm_es_acc"], res_mgsm_es = evaluate_mgsm(mgsm_es, lang='es', n_samples=None)
    summary["mgsm_ur_acc"], res_mgsm_ur = evaluate_mgsm(mgsm_ur, lang='ur', n_samples=None)
    print('MGSM Done - Lang')

    summary["xnli_zh_acc"], res_xnli_zh = evaluate_xnli(xnli_zh, lang='zh', n_samples=None)
    summary["xnli_es_acc"], res_xnli_es = evaluate_xnli(xnli_es, lang='es', n_samples=None)
    summary["xnli_ur_acc"], res_xnli_ur = evaluate_xnli(xnli_ur, lang='ur', n_samples=None)
    print('XNLI Done - Lang')

    summary["csqa_zh_acc"], res_csqa_zh = evaluate_csqa(csqa_zh, lang='zh', n_samples=None)
    summary["csqa_es_acc"], res_csqa_es = evaluate_csqa(csqa_es, lang='es', n_samples=None)
    summary["csqa_ur_acc"], res_csqa_ur = evaluate_csqa(csqa_ur, lang='ur', n_samples=None)
    print('CSQA Done - Lang')

    return {
        "summary": summary,
        "mgsm_zh": res_mgsm_zh,
        "mgsm_es": res_mgsm_es,
        "mgsm_ur": res_mgsm_ur,
        "xnli_zh": res_xnli_zh,
        "xnli_es": res_xnli_es,
        "xnli_ur": res_xnli_ur,
        "csqa_zh": res_csqa_zh,
        "csqa_es": res_csqa_es,
        "csqa_ur": res_csqa_ur,
    }


def save_results(results, summary_path, full_path):
    json_summary = {
        "summary": results["summary"]
    }

    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(json_summary, f, ensure_ascii=False, indent=2)

    json_results = {
        "summary": results["summary"],
    }
    for key, value in results.items():
        if key == "summary":
            continue
        json_results[key] = value.to_dict(orient="records")

    with open(full_path, "w", encoding="utf-8") as f:
        json.dump(json_results, f, ensure_ascii=False, indent=2)


if __name__ == "__main__":
    english_results = eval_english_prompts()
    language_results = eval_native_prompts()

    english_summary_path = f"/kaggle/working/{cond}_results_summary_english.json"
    english_full_path = f"/kaggle/working/{cond}_results_english.json"

    language_summary_path = f"/kaggle/working/{cond}_results_summary_language.json"
    language_full_path = f"/kaggle/working/{cond}_results_language.json"

    save_results(english_results, english_summary_path, english_full_path)
    save_results(language_results, language_summary_path, language_full_path)

    api = HfApi()

    # Upload results to HuggingFace
    # File schema:
    #   english_prompt_results.json — English prompts + zh/es/ur/en data (12 metrics)
    #     summary: {mgsm,xnli,csqa}_{zh,es,ur,en}_acc
    #     per-example: mgsm_{zh,es,ur,en}, xnli_{zh,es,ur,en}, csqa_{zh,es,ur,en}
    #   native_prompt_results.json — Native prompts + zh/es/ur data (9 metrics)
    #     summary: {mgsm,xnli,csqa}_{zh,es,ur}_acc
    #     per-example: mgsm_{zh,es,ur}, xnli_{zh,es,ur}, csqa_{zh,es,ur}
    for local_path, path_in_repo in [
        (english_full_path, f"conditions/{cond}/results/english_prompt_results.json"),
        (language_full_path, f"conditions/{cond}/results/native_prompt_results.json"),
    ]:
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=path_in_repo,
            repo_id="legesher/language-decoded-experiments",
            repo_type="dataset",
        )

MGSM Done - EN
XNLI Done - EN
CSQA Done - EN
MGSM Done - Lang
XNLI Done - Lang
CSQA Done - Lang
